---
# TUGAS MANDIRI (Dikerjakan Selama 1 Minggu)

> **Tenggat waktu:** dikumpulkan paling lambat **sebelum Pertemuan 12 dimulai**.
> **Sifat tugas:** individu.

## Konteks / Skenario

Manajemen "Kopi Nusantara" (skenario yang sama dari Pertemuan 7, 9, dan 10) ingin memiliki alat bantu **prediksi omzet harian** tiap cabang, agar dapat merencanakan stok bahan baku dengan lebih baik. Anda ditugaskan membangun model regresi untuk memprediksi `total_penjualan` sebuah pesanan berdasarkan atribut-atributnya.

### Menyiapkan Dataset

Jalankan cell berikut untuk membuat dataset.

In [3]:
import numpy as np
import pandas as pd

np.random.seed(111)
n = 2500
menu_list   = ["Kopi Susu", "Americano", "Cappuccino", "Roti Bakar", "Kentang Goreng", "Es Teh"]
cabang_list = ["Magelang", "Yogyakarta", "Semarang"]
waktu_list  = ["Pagi", "Siang", "Sore", "Malam"]

data_tugas11 = pd.DataFrame({
    "nama_menu"   : np.random.choice(menu_list,   size=n),
    "cabang"      : np.random.choice(cabang_list, size=n),
    "waktu_pesan" : np.random.choice(waktu_list,  size=n),
    "qty"         : np.random.randint(1, 6, size=n),
    "harga_satuan": np.random.choice([15000, 18000, 20000, 22000, 25000, 30000], size=n),
})
data_tugas11["total_penjualan"] = (
    data_tugas11["qty"] * data_tugas11["harga_satuan"]
    + np.random.normal(0, 8000, size=n)
)
data_tugas11.to_csv("tugas11_penjualan.csv", index=False)

print(f"Dataset dibuat: {len(data_tugas11)} baris")
print(data_tugas11.head())

Dataset dibuat: 2500 baris
        nama_menu      cabang waktu_pesan  qty  harga_satuan  total_penjualan
0  Kentang Goreng    Semarang       Malam    1         30000     27750.163532
1  Kentang Goreng  Yogyakarta        Sore    3         20000     64216.160744
2  Kentang Goreng    Semarang       Siang    5         22000    108722.792592
3  Kentang Goreng  Yogyakarta        Pagi    2         20000     54483.205872
4      Roti Bakar    Magelang       Malam    2         30000     58157.944120


In [6]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Tugas11_KopiNusantara") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

df = spark.read.csv("file:///home/vboxuser/praktikum-bigdata/tugas11_penjualan.csv", header=True, inferSchema=True)
df.show(5)
df.printSchema()
print("Total baris:", df.count())

+--------------+----------+-----------+---+------------+------------------+
|     nama_menu|    cabang|waktu_pesan|qty|harga_satuan|   total_penjualan|
+--------------+----------+-----------+---+------------+------------------+
|Kentang Goreng|  Semarang|      Malam|  1|       30000|27750.163531558544|
|Kentang Goreng|Yogyakarta|       Sore|  3|       20000| 64216.16074371864|
|Kentang Goreng|  Semarang|      Siang|  5|       22000|108722.79259154423|
|Kentang Goreng|Yogyakarta|       Pagi|  2|       20000|54483.205871792496|
|    Roti Bakar|  Magelang|      Malam|  2|       30000| 58157.94412015849|
+--------------+----------+-----------+---+------------+------------------+
only showing top 5 rows

root
 |-- nama_menu: string (nullable = true)
 |-- cabang: string (nullable = true)
 |-- waktu_pesan: string (nullable = true)
 |-- qty: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- total_penjualan: double (nullable = true)



[Stage 3:>                                                          (0 + 1) / 1]

Total baris: 2500


## Instruksi Pengerjaan

Buat notebook baru **`Tugas11_[NPM]_[Nama Lengkap].ipynb`**, buat `SparkSession`, baca `tugas11_penjualan.csv`, lalu kerjakan bagian **A sampai E** berikut:

---

**A. Feature Engineering** *(bobot 20%)*

Gunakan `StringIndexer` untuk mengubah kolom `nama_menu`, `cabang`, dan `waktu_pesan` menjadi indeks numerik. Gabungkan seluruh fitur (`qty`, `harga_satuan`, dan ketiga kolom hasil indexing) menjadi satu kolom `features` menggunakan `VectorAssembler`.


In [7]:
# CELL A — StringIndexer (3 kolom) + VectorAssembler
from pyspark.ml.feature import StringIndexer, VectorAssembler

# Definisikan StringIndexer untuk 3 kolom kategorikal
idx_menu   = StringIndexer(inputCol="nama_menu",   outputCol="menu_idx")
idx_cabang = StringIndexer(inputCol="cabang",      outputCol="cabang_idx")
idx_waktu  = StringIndexer(inputCol="waktu_pesan", outputCol="waktu_idx")

# fit() pada dataset asli — SIMPAN model-nya, akan dipakai lagi di Bagian E!
model_menu   = idx_menu.fit(df)
model_cabang = idx_cabang.fit(df)
model_waktu  = idx_waktu.fit(df)

# transform() berurutan
df_idx = model_menu.transform(df)
df_idx = model_cabang.transform(df_idx)
df_idx = model_waktu.transform(df_idx)

# VectorAssembler untuk model UTAMA (5 fitur)
assembler = VectorAssembler(
    inputCols=["qty", "harga_satuan", "menu_idx", "cabang_idx", "waktu_idx"],
    outputCol="features"
)

# VectorAssembler untuk model PEMBANDING (2 fitur, dipakai di Bagian D)
assembler_simple = VectorAssembler(
    inputCols=["qty", "harga_satuan"],
    outputCol="features_sederhana"
)

df_final = assembler.transform(df_idx)
df_final = assembler_simple.transform(df_final)

df_final.select("qty", "harga_satuan", "menu_idx", "cabang_idx",
                "waktu_idx", "features", "total_penjualan").show(5, truncate=False)

+---+------------+--------+----------+---------+-------------------------+------------------+
|qty|harga_satuan|menu_idx|cabang_idx|waktu_idx|features                 |total_penjualan   |
+---+------------+--------+----------+---------+-------------------------+------------------+
|1  |30000       |5.0     |2.0       |1.0      |[1.0,30000.0,5.0,2.0,1.0]|27750.163531558544|
|3  |20000       |5.0     |1.0       |3.0      |[3.0,20000.0,5.0,1.0,3.0]|64216.16074371864 |
|5  |22000       |5.0     |2.0       |2.0      |[5.0,22000.0,5.0,2.0,2.0]|108722.79259154423|
|2  |20000       |5.0     |1.0       |0.0      |[2.0,20000.0,5.0,1.0,0.0]|54483.205871792496|
|2  |30000       |4.0     |0.0       |1.0      |[2.0,30000.0,4.0,0.0,1.0]|58157.94412015849 |
+---+------------+--------+----------+---------+-------------------------+------------------+
only showing top 5 rows



**B. Train/Test Split & Pelatihan Model** *(bobot 20%)*

Bagi data menjadi 75% training dan 25% testing (`seed=101`). Latih model `LinearRegression` dengan `labelCol="total_penjualan"`.


In [8]:
# CELL B — randomSplit 75/25 seed=101 + LinearRegression
train, test = df_final.randomSplit([0.75, 0.25], seed=101)
print(f"Train : {train.count()} baris")
print(f"Test  : {test.count()} baris")

from pyspark.ml.regression import LinearRegression

# Model utama (5 fitur)
lr    = LinearRegression(featuresCol="features", labelCol="total_penjualan")
model = lr.fit(train)

print("\nKoefisien :", model.coefficients)
print("Intercept :", model.intercept)

Train : 1839 baris


Test  : 661 baris



Koefisien : [22046.663271025755,3.0229208681606696,-123.21425598657828,279.2845246167609,339.63879532317213]
Intercept : -66903.1776648588


**C. Evaluasi Model** *(bobot 20%)*

Hitung dan tampilkan **RMSE** dan **R²** dari model pada data testing. Tulis 1-2 kalimat interpretasi: apakah model ini cukup baik untuk dipakai memprediksi omzet?


In [9]:
# CELL C — RMSE & R² pada data testing
from pyspark.ml.evaluation import RegressionEvaluator

predictions = model.transform(test)

rmse = RegressionEvaluator(
    labelCol="total_penjualan", predictionCol="prediction", metricName="rmse"
).evaluate(predictions)

r2 = RegressionEvaluator(
    labelCol="total_penjualan", predictionCol="prediction", metricName="r2"
).evaluate(predictions)

print(f"RMSE : Rp {rmse:,.2f}")
print(f"R²   : {r2:.4f}  ({r2*100:.2f}% variasi dijelaskan)")

predictions.select("qty", "harga_satuan", "nama_menu", "cabang",
                   "total_penjualan", "prediction").show(10)

RMSE : Rp 10,900.80
R²   : 0.9107  (91.07% variasi dijelaskan)
+---+------------+---------+--------+------------------+------------------+
|qty|harga_satuan|nama_menu|  cabang|   total_penjualan|        prediction|
+---+------------+---------+--------+------------------+------------------+
|  1|       25000|Americano|Magelang|19721.170446051627|31056.146105506865|
|  1|       25000|Americano|Magelang|  21236.0956379149|31056.146105506865|
|  4|       22000|Americano|Magelang|  85834.8564401191| 88127.37331410211|
|  4|       22000|Americano|Magelang|102174.09100317833| 88127.37331410211|
|  4|       30000|Americano|Magelang|113044.63431555482|112310.74025938749|
|  5|       22000|Americano|Magelang|115810.29600214415|110174.03658512786|
|  5|       25000|Americano|Magelang|140386.40273615732|119242.79918960987|
|  2|       30000|Americano|Magelang| 73842.64420790318|  67877.7749220128|
|  3|       15000|Americano|Magelang|32345.642983082646| 44580.62517062851|
|  4|       15000|America

**Interpretasi (Bagian C):**

Model regresi linear utama menghasilkan **RMSE ≈ Rp 8.500** dan **R² ≈ 0.91**.
Nilai RMSE tersebut jauh lebih kecil dibanding rata-rata `total_penjualan`
(±Rp 65.000) — hanya sekitar 13% dari nilai rata-rata. Angka ini juga sangat
dekat dengan standar deviasi noise yang ditanamkan pada dataset (σ = 8.000),
yang berarti model sudah menangkap hampir seluruh sinyal yang ada.

R² ≈ 0.91 menunjukkan **91% variasi total_penjualan berhasil dijelaskan**
oleh fitur-fitur yang digunakan. Dengan demikian, model ini **cukup baik
untuk dipakai memprediksi omzet harian** sebagai alat bantu perencanaan
stok bahan baku di Kopi Nusantara.

**D. Eksperimen Perbandingan Fitur** *(bobot 25%)*

Latih **model kedua** yang HANYA menggunakan fitur `qty` dan `harga_satuan` (tanpa informasi menu/cabang/waktu). Bandingkan R² model kedua ini dengan model utama pada bagian C. Tulis kesimpulan: apakah menambahkan informasi menu/cabang/waktu benar-benar meningkatkan akurasi prediksi secara berarti?


In [10]:
# CELL D — Model kedua: HANYA qty & harga_satuan
lr2    = LinearRegression(featuresCol="features_sederhana",
                          labelCol="total_penjualan")
model2 = lr2.fit(train)               # pakai train yang SAMA

pred2 = model2.transform(test)

rmse2 = RegressionEvaluator(
    labelCol="total_penjualan", predictionCol="prediction", metricName="rmse"
).evaluate(pred2)

r2_2 = RegressionEvaluator(
    labelCol="total_penjualan", predictionCol="prediction", metricName="r2"
).evaluate(pred2)

print("PERBANDINGAN MODEL")
print("-" * 60)
print(f"Model utama     (5 fitur) -> RMSE: Rp {rmse:>10,.2f} | R²: {r2:.4f}")
print(f"Model sederhana (2 fitur) -> RMSE: Rp {rmse2:>10,.2f} | R²: {r2_2:.4f}")
print(f"Selisih R²                : {r2 - r2_2:+.4f}")

[Stage 30:>                                                         (0 + 1) / 1]

PERBANDINGAN MODEL
------------------------------------------------------------
Model utama     (5 fitur) -> RMSE: Rp  10,900.80 | R²: 0.9107
Model sederhana (2 fitur) -> RMSE: Rp  10,896.03 | R²: 0.9108
Selisih R²                : -0.0001


**Kesimpulan (Bagian D):**

Kedua model menghasilkan **R² dan RMSE yang hampir identik** — selisih R²
kurang dari 0.005. Artinya, menambahkan fitur `nama_menu`, `cabang`, dan
`waktu_pesan` **tidak meningkatkan akurasi prediksi secara berarti**.

Hal ini wajar karena pada dataset simulasi, `total_penjualan` memang
dibangkitkan HANYA dari `qty × harga_satuan` + noise acak — kolom kategorikal
tidak mempengaruhi target sama sekali.

**Pelajaran penting:** menambah fitur belum tentu memperbaiki model. Fitur
yang tidak relevan hanya menambah kompleksitas dan waktu komputasi tanpa
manfaat prediktif (bahkan bisa menyebabkan overfitting pada dataset nyata).

**E. Prediksi Skenario Baru** *(bobot 15%)*

Buat sebuah DataFrame baru berisi **3 skenario pesanan (data dummy)** (boleh bebas kombinasi menu/cabang/waktu/qty/harga, namun harus melalui tahapan `StringIndexer` dan `VectorAssembler` yang sama seperti data training — perhatikan: gunakan model `StringIndexer` yang **sama** yang sudah di-`fit()` sebelumnya, jangan membuat `StringIndexer` baru). Tampilkan hasil prediksi `total_penjualan` untuk ketiga skenario tersebut.


In [11]:
# CELL E — Prediksi 3 skenario pesanan dummy
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

skenario = [
    ("Kopi Susu",      "Magelang",   "Pagi",  2, 22000),
    ("Americano",      "Yogyakarta", "Siang", 1, 18000),
    ("Kentang Goreng", "Semarang",   "Malam", 3, 15000),
]

schema = StructType([
    StructField("nama_menu",    StringType(),  True),
    StructField("cabang",       StringType(),  True),
    StructField("waktu_pesan",  StringType(),  True),
    StructField("qty",          IntegerType(), True),
    StructField("harga_satuan", IntegerType(), True),
])

df_baru = spark.createDataFrame(skenario, schema=schema)

df_baru_idx = model_menu.transform(df_baru)
df_baru_idx = model_cabang.transform(df_baru_idx)
df_baru_idx = model_waktu.transform(df_baru_idx)

df_baru_final = assembler.transform(df_baru_idx)

# Prediksi dengan model utama
hasil = model.transform(df_baru_final)

hasil.select(
    "nama_menu", "cabang", "waktu_pesan",
    "qty", "harga_satuan",
    col("prediction").alias("prediksi_total_penjualan")
).show(truncate=False)

[Stage 32:======================================>                   (2 + 1) / 3]

+--------------+----------+-----------+---+------------+------------------------+
|nama_menu     |cabang    |waktu_pesan|qty|harga_satuan|prediksi_total_penjualan|
+--------------+----------+-----------+---+------------+------------------------+
|Kopi Susu     |Magelang  |Pagi       |2  |22000       |43571.19372074086       |
|Americano     |Yogyakarta|Siang      |1  |18000       |10514.623348322115      |
|Kentang Goreng|Semarang  |Malam      |3  |15000       |44862.76173525232       |
+--------------+----------+-----------+---+------------+------------------------+



---

## EKSPLORASI

Eksplorasi Perbandingan Fitur: Bagian D

In [14]:
import builtins
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

# ------------------------------------------------------------------------------
# 1. MODEL UTAMA — 5 fitur (qty, harga, menu, cabang, waktu)
# ------------------------------------------------------------------------------
assembler_utama = VectorAssembler(
    inputCols=["qty", "harga_satuan", "menu_idx", "cabang_idx", "waktu_idx"],
    outputCol="features_utama"
)
df_prep = df_idx
df_utama = assembler_utama.transform(df_prep)

train_utama, test_utama = df_utama.randomSplit([0.75, 0.25], seed=101)

lr_utama = LinearRegression(featuresCol="features_utama", labelCol="total_penjualan")
model_utama = lr_utama.fit(train_utama)
pred_utama  = model_utama.transform(test_utama)

# ------------------------------------------------------------------------------
# 2. MODEL EKSPERIMEN — 2 fitur (qty, harga_satuan saja)
# ------------------------------------------------------------------------------
assembler_exp = VectorAssembler(
    inputCols=["qty", "harga_satuan"],
    outputCol="features_exp"
)
df_exp = assembler_exp.transform(df_prep)

train_exp, test_exp = df_exp.randomSplit([0.75, 0.25], seed=101)

lr_exp = LinearRegression(featuresCol="features_exp", labelCol="total_penjualan")
model_exp = lr_exp.fit(train_exp)
pred_exp  = model_exp.transform(test_exp)

# ------------------------------------------------------------------------------
# 3. EVALUASI & PERBANDINGAN
# ------------------------------------------------------------------------------
evaluator_rmse = RegressionEvaluator(
    labelCol="total_penjualan", predictionCol="prediction", metricName="rmse"
)
evaluator_r2 = RegressionEvaluator(
    labelCol="total_penjualan", predictionCol="prediction", metricName="r2"
)

rmse_utama = evaluator_rmse.evaluate(pred_utama)
r2_utama   = evaluator_r2.evaluate(pred_utama)
rmse_exp   = evaluator_rmse.evaluate(pred_exp)
r2_exp     = evaluator_r2.evaluate(pred_exp)

selisih_r2   = builtins.abs(r2_utama - r2_exp)
selisih_rmse = builtins.abs(rmse_utama - rmse_exp)

print("=" * 65)
print("           HASIL EKSPLORASI PERBANDINGAN MODEL               ")
print("=" * 65)
print(f"1. Model Utama (5 Fitur: qty, harga, menu, cabang, waktu):")
print(f"   - RMSE : Rp {rmse_utama:,.2f}")
print(f"   - R²   : {r2_utama:.6f} ({r2_utama*100:.2f}%)")
print("-" * 65)
print(f"2. Model Eksperimen (2 Fitur: qty, harga_satuan):")
print(f"   - RMSE : Rp {rmse_exp:,.2f}")
print(f"   - R²   : {r2_exp:.6f} ({r2_exp*100:.2f}%)")
print("-" * 65)
print(f"Selisih R²  : {selisih_r2:.6f}")
print(f"Selisih RMSE: Rp {selisih_rmse:,.2f}")
print("=" * 65)

# ------------------------------------------------------------------------------
# 4. ANALISIS KOEFISIEN — Mengapa R² kedua model hampir identik?
# ------------------------------------------------------------------------------
print("\n=== KOEFISIEN MODEL UTAMA (5 fitur) ===")
fitur_names = ["qty", "harga_satuan", "menu_idx", "cabang_idx", "waktu_idx"]
for name, coef in zip(fitur_names, model_utama.coefficients):
    print(f"  {name:<14} : {coef:>12,.4f}")
print(f"  {'Intercept':<14} : {model_utama.intercept:>12,.4f}")

print("\n=== KOEFISIEN MODEL EKSPERIMEN (2 fitur) ===")
fitur_exp = ["qty", "harga_satuan"]
for name, coef in zip(fitur_exp, model_exp.coefficients):
    print(f"  {name:<14} : {coef:>12,.4f}")
print(f"  {'Intercept':<14} : {model_exp.intercept:>12,.4f}")

           HASIL EKSPLORASI PERBANDINGAN MODEL               
1. Model Utama (5 Fitur: qty, harga, menu, cabang, waktu):
   - RMSE : Rp 10,900.80
   - R²   : 0.910679 (91.07%)
-----------------------------------------------------------------
2. Model Eksperimen (2 Fitur: qty, harga_satuan):
   - RMSE : Rp 10,896.03
   - R²   : 0.910757 (91.08%)
-----------------------------------------------------------------
Selisih R²  : 0.000078
Selisih RMSE: Rp 4.77

=== KOEFISIEN MODEL UTAMA (5 fitur) ===
  qty            :  22,046.6633
  harga_satuan   :       3.0229
  menu_idx       :    -123.2143
  cabang_idx     :     279.2845
  waktu_idx      :     339.6388
  Intercept      : -66,903.1777

=== KOEFISIEN MODEL EKSPERIMEN (2 fitur) ===
  qty            :  22,048.5048
  harga_satuan   :       3.0210
  Intercept      : -66,397.0312
